[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/48_adaln_zero_modulation_solution.ipynb)

# 🟡 Solution: AdaLN-Zero Modulation

Reference solution for `adaln_zero_modulation`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn


In [ ]:
# ✅ SOLUTION

class AdaLNZero(nn.Module):
    def __init__(self, dim: int, cond_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.modulation = nn.Linear(cond_dim, 6 * dim)
        nn.init.zeros_(self.modulation.weight)
        nn.init.zeros_(self.modulation.bias)

    def _modulate(self, x, shift, scale):
        return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

    def forward(self, x: torch.Tensor, cond: torch.Tensor):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.modulation(cond).chunk(6, dim=-1)
        x_norm = self.norm(x)
        x_msa = self._modulate(x_norm, shift_msa, scale_msa)
        x_mlp = self._modulate(x_norm, shift_mlp, scale_mlp)
        return x_msa, gate_msa.unsqueeze(1), x_mlp, gate_mlp.unsqueeze(1)


In [ ]:
# Verify
mod = AdaLNZero(dim=8, cond_dim=4)
x = torch.randn(2, 5, 8)
cond = torch.randn(2, 4)
x_msa, gate_msa, x_mlp, gate_mlp = mod(x, cond)
print(x_msa.shape, gate_msa.shape, x_mlp.shape, gate_mlp.shape)
print("initial gates:", gate_msa.abs().max().item(), gate_mlp.abs().max().item())


In [ ]:
# Run judge
from torch_judge import check
check('adaln_zero_modulation')
